# AMD/NVIDIA SEC Item 2.02 原始季度公告清点

冻结 133 个 2011–2026 官方 8-K 候选，逐份读取完整提交原件并 NAS 备份，只做来源身份与收入展望初步分类。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,re
from pathlib import Path
from datetime import datetime,timezone
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';plan=json.loads((card/'item202_inventory_plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'item202_inventory_state.json').exists()
for name,key in [('result.json','four_anchor'),('main_submissions_result.json','main_submissions'),('older_repair_result.json','older_repair')]:assert sha(card/name)==plan['parent_results_sha256'][key]
for name,h in plan['raw_index_sha256'].items():assert sha(raw/name)==h
candidates=[]
for issuer,cik in [('NVDA','0001045810'),('AMD','0000002488')]:
 for part in ('main','older'):
  d=json.loads((raw/f'{issuer.lower()}_{part}_submissions.json').read_text());s=d['filings']['recent'] if part=='main' else d
  for i,form in enumerate(s['form']):
   day=s['filingDate'][i]
   if form=='8-K' and '2011-01-01'<=day<='2026-10-23' and '2.02' in str(s['items'][i]):
    candidates.append({'issuer':issuer,'cik':cik,'filed':day,'accepted_api':s['acceptanceDateTime'][i],'accession':s['accessionNumber'][i],'metadata_size':s['size'][i]})
assert len(candidates)==133 and len({c['accession'] for c in candidates})==133 and sum(c['issuer']=='NVDA' for c in candidates)==66 and sum(c['issuer']=='AMD' for c in candidates)==67
candidates.sort(key=lambda c:(c['issuer'],c['filed'],c['accepted_api'],c['accession']))
state={'status':'started','GET_attempts':0,'processed':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'item202_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[];checks=[];failure=None
for c in candidates:
 acc=c['accession'];body=raw/f'item202_{acc}.txt';headers=raw/f'item202_{acc}.headers';assert not body.exists();state['GET_attempts']+=1;(card/'item202_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
 url=plan['exact_request_template'].format(cik_integer=int(c['cik']),accession_no_hyphen=acc.replace('-',''),accession=acc)
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','60','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',url]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'issuer':c['issuer'],'accession':acc,'filed':c['filed'],'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:250]}
 except Exception as exc:receipt={'issuer':c['issuer'],'accession':acc,'filed':c['filed'],'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:250]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None);receipts.append(receipt);(card/'item202_inventory_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=acc+'_transport';break
 try:
  b=body.read_bytes();head=b[:15000].decode('utf8','replace');assert '<SEC-DOCUMENT>' in head.upper() and acc in head
  cik=re.search(r'CENTRAL INDEX KEY:\s*0*(\d+)',head);accepted=re.search(r'ACCEPTANCE-DATETIME>\s*(\d{14})',head);assert cik and cik.group(1)==str(int(c['cik'])) and accepted
  parts=[]
  for doc in re.findall(rb'<DOCUMENT>(.*?)</DOCUMENT>',b,re.S|re.I):
   typ=re.search(rb'<TYPE>\s*([^\r\n<]+)',doc,re.I);content=re.search(rb'<TEXT>(.*?)</TEXT>',doc,re.S|re.I)
   if typ and content and typ.group(1).strip().upper() in (b'8-K',b'EX-99.1',b'EX99.1',b'EX-99'):
    parts.append(BeautifulSoup(content.group(1).decode('utf8','replace'),'html.parser').get_text(' ',strip=True))
  text=re.sub(r'\s+',' ',' '.join(parts));name='NVIDIA' if c['issuer']=='NVDA' else 'AMD';title=re.search(r'\b'+name+r'\b.{0,120}\b(First|Second|Third|Fourth)\s+Quarter\b',text,re.I);outlook=re.search(r'\b(?:current\s+)?outlook\b',text,re.I);section=text[outlook.start():outlook.start()+1600] if outlook else '';revenue=re.search(r'\brevenue\b.{0,200}',section,re.I);has_guidance=bool(title and outlook and revenue and re.search(r'\b(expect|forecast)',section,re.I))
  checks.append({'issuer':c['issuer'],'accession':acc,'filed':c['filed'],'accepted_header':accepted.group(1),'raw_sha256':sha(body),'text_chars':len(text),'original_quarter_results_and_forward_revenue_outlook':has_guidance,'quarter_title':title.group(0)[:150] if title else None,'revenue_outlook_excerpt':revenue.group(0)[:220] if revenue else None});state['processed']+=1;(card/'item202_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');(card/'item202_inventory_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
 except Exception as exc:failure=acc+'_identity_'+type(exc).__name__+'_'+str(exc)[:100];break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.name.startswith('item202_')];manifest={'plan_sha256':sha(card/'item202_inventory_plan.json'),'files':files,'receipts_sha256':sha(card/'item202_inventory_receipts.json'),'checks_sha256':sha(card/'item202_inventory_checks.json') if (card/'item202_inventory_checks.json').exists() else None};(card/'item202_inventory_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='amd-nvda-item202-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('item202_inventory_plan.json','item202_inventory_receipts.json','item202_inventory_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='amd-nvda-item202-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=300)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
by_issuer={issuer:{'item202':sum(x['issuer']==issuer for x in checks),'quarter_outlook_matches':sum(x['issuer']==issuer and x['original_quarter_results_and_forward_revenue_outlook'] for x in checks)} for issuer in ('AMD','NVDA')}
result={'status':'AMD_NVDA_2011_2026_item202_originals_captured_quarter_outlook_identity_pending' if not failure and len(checks)==133 else 'bounded_AMD_NVDA_item202_inventory_partial_or_failed','failure':failure,'plan_sha256':sha(card/'item202_inventory_plan.json'),'GET_attempts':state['GET_attempts'],'processed':len(checks),'by_issuer':by_issuer,'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'quarter_event_completeness_qualified':False,'numeric_harmonization_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'item202_inventory_result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if not failure else 'partial_or_failed',NAS_archive_sha256=digest);(card/'item202_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'by_issuer':by_issuer,'NAS':digest})


{'status': 'AMD_NVDA_2011_2026_item202_originals_captured_quarter_outlook_identity_pending', 'failure': None, 'GETs': 133, 'by_issuer': {'AMD': {'item202': 67, 'quarter_outlook_matches': 61}, 'NVDA': {'item202': 66, 'quarter_outlook_matches': 61}}, 'NAS': '8273fd8c00608f5b2f1faf81368b9f3c238919f297b5d868c8af319b115d9cee'}


## 中文结论

133 份 Item2.02 原始 8-K 与头部哈希/NAS 全部核对；初筛 AMD/NVIDIA 各 61 份数字指引，11 份未匹配另行复核。 完整决定见 `conclusion.md`。